# Run 3 (2024) data / MC comparison

Same plots and checks as the Run 2 `DataVsMC.ipynb`: stacked MC with the stat ⊕ weight-systematic
band, data with blinding, a 3×3 grid of variables per PDF page, the raw-count table, the non-bb
signal check, the pileup variations and the Xbb S/√B scan.

Inputs: the merged inference outputs (`outputs/nominal/<group>/output_all.coffea`), read one file at
a time. MC weight = `norm_weight` (xsec × lumi) × sign(genWeight) / Σ sign(genWeight) × the SFs of
`outvars.nominal_weight_factors`. tt+B is taken from the 4FS TTbb samples and tt+LF/C from the 5FS
TTTo samples. Until they are derived, the electron trigger SF and the b-tag SF are 1 and the muon
trigger SF is the MUO reference.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mplhep as hep
from matplotlib.backends.backend_pdf import PdfPages
from coffea.util import load

import outvars
from analysis_years import LUMI_FB, XBB_WP_DICT

# ==========================================
# 1. GLOBAL SETTINGS & CONFIGURATION
# ==========================================
hep.style.use(hep.style.CMS)

# ---------------- USER SWITCHES ----------------
TARGET_ERA = "2024"

# Xbb (ZH_bbvLscore) cut: None (no cut), "LP", "MP" or "HP" (needs analysis_years.XBB_WP_DICT)
XBB_WP = None

# WP used by analysis_cuts() in the non-bb signal check; None = no Xbb cut (no Run 3 WPs yet)
ANALYSIS_XBB_WP = None

CHANNEL = "all"              # "all", "ele" or "mu" (flavour of the tight lepton)
INCLUDE_QCD = False          # negligible, not part of the analysis
TTB_SCALE = 1.0              # extra scale of the 4FS tt+B yield (1 = dataset cross sections)
BLIND = {"ZH_bbvLscore": 0.8}   # data hidden for bin centers above these values
# -----------------------------------------------

COFFEA_DIR = "outputs/nominal"          # run_pipeline.py layout: <dir>/<group>/output_all.coffea
GROUPS = ["signal", "bkg", "minor", "data"] + (["qcd"] if INCLUDE_QCD else [])
CATEGORY = "btag_mask"

LUMI = LUMI_FB[TARGET_ERA]
for wp_name, wp in [("XBB_WP", XBB_WP), ("ANALYSIS_XBB_WP", ANALYSIS_XBB_WP)]:
    if wp not in (None, "LP", "MP", "HP"):
        raise ValueError(f"Unknown {wp_name} '{wp}'. Choose None, 'LP', 'MP' or 'HP'")
    if wp is not None and TARGET_ERA not in XBB_WP_DICT:
        raise ValueError(f"{wp_name}={wp}, but there are no Xbb working points for {TARGET_ERA} yet")

XBB_TAG = f"Xbb{XBB_WP}" if XBB_WP else "NoXbb"
OUTPUT_TAG = f"{TARGET_ERA}_{CHANNEL}_{XBB_TAG}"

print("=" * 50)
print(f" Era        : {TARGET_ERA}")
print(f" Lumi       : {LUMI} fb^-1")
print(f" Input dir  : {COFFEA_DIR} ({', '.join(GROUPS)})")
print(f" Channel    : {CHANNEL}")
print(f" Xbb cut    : {XBB_WP_DICT[TARGET_ERA][XBB_WP] if XBB_WP else 'none'}")
print(f" Output tag : {OUTPUT_TAG}")
print("=" * 50)


def xbb_threshold(wp):
    return XBB_WP_DICT[TARGET_ERA][wp]


def cuts(df_):
    """Validation selection (on top of the btag_mask category), as in the Run 2 notebook."""
    base = (
        (df_['n_ak4jets'] >= 5) &
        (df_['n_b_outZH'] >= 2) &
        (df_['ZH_pt'] >= 200) &
        (df_['ZH_M'] >= 50) &
        (df_['ZH_M'] <= 200)
    )
    if XBB_WP is not None:
        base &= (df_['ZH_bbvLscore'] > xbb_threshold(XBB_WP))
    if CHANNEL == "ele":
        base &= (df_['ele_pt'] > 0)
    elif CHANNEL == "mu":
        base &= (df_['muon_pt'] > 0)
    return base


def analysis_cuts(df_):
    """Datacard selection (applied on top of the validation cuts)."""
    sel = (df_['n_b_outZH'] == 2) & (df_['MET_pt'] >= 20)
    if ANALYSIS_XBB_WP is not None:
        sel &= (df_['ZH_bbvLscore'] >= xbb_threshold(ANALYSIS_XBB_WP))
    return sel

In [ ]:
# ==========================================
# 2. PROCESSES, VARIABLES AND WEIGHTS
# ==========================================
bkg_processes = ['VJets', 'tt_B', 'TTBar', 'SingleTop', 'TTX', 'VV'] + (['QCD'] if INCLUDE_QCD else [])
sig_processes = ['ttZ', 'ttH']
data_process = 'data_obs'
all_processes = bkg_processes + sig_processes + [data_process]

bkg_colors = {'VJets': '#3f90da', 'QCD': '#ffa90e', 'tt_B': '#bd1f01', 'TTBar': '#94a4a2',
              'SingleTop': '#e76300', 'TTX': '#b9ac70', 'VV': '#717581'}
sig_colors = {'ttZ': '#832db6', 'ttH': '#a96b59'}
mc_colors = {**bkg_colors, **sig_colors}

process_labels = {
    'VJets': 'V+Jets', 'QCD': 'QCD', 'tt_B': r'$t\bar{t}+bb$',
    'TTBar': r'$t\bar{t} + lf, t\bar{t}+cc$', 'SingleTop': 'single top',
    'TTX': r'$t\bar{t}W, t\bar{t}t\bar{t}, tHW, tHq, tZq$', 'VV': 'VV',
    'ttZ': r'$t\bar{t}Z$', 'ttH': r'$t\bar{t}H$', 'data_obs': 'Data',
}


def map_process(key):
    """Output key (sample or sample__subsample) -> process; None = not used.
    tt+B from the 4FS TTbb samples, tt+LF/C from the 5FS TTTo samples (no double counting)."""
    sample, _, sub = key.partition("__")
    if sample.startswith("DATA"):
        return 'data_obs'
    if sample.startswith("TTbb"):
        return 'tt_B' if sub == "tt+B" else None
    if sample.startswith("TTTo"):
        return 'TTBar' if sub in ("tt+LF", "tt+C") else None
    mapping = {"ttHTobb": "ttH", "ttHToNonbb": "ttH", "TTZToQQ": "ttZ", "TTLL": "ttZ", "TTNuNu": "ttZ",
               "WJets": "VJets", "DYJets": "VJets", "SingleTop": "SingleTop", "TTX": "TTX", "VV": "VV",
               "QCD": "QCD"}
    proc = mapping.get(sample)
    return proc if proc in all_processes else None


# --- Weights (2024: trigger SFs are 1 / MUO reference and btag_sf is 1 until derived) ---
NOMINAL_FACTORS = list(outvars.nominal_weight_factors)
weight_syst_mapping = {
    'topptWeight': ('topptWeight', 'topptWeight_Up', 'topptWeight_Down'),
    'ele_reco_sf': ('ele_reco_sf', 'ele_reco_sfup', 'ele_reco_sfdown'),
    'ele_id_sf':   ('ele_id_sf', 'ele_id_sfup', 'ele_id_sfdown'),
    'ele_trig_sf': ('ele_trig_sf', 'ele_trig_sfup', 'ele_trig_sfdown'),
    'mu_id_sf':    ('mu_id_sf', 'mu_id_sfup', 'mu_id_sfdown'),
    'mu_iso_sf':   ('mu_iso_sf', 'mu_iso_sfup', 'mu_iso_sfdown'),
    'mu_trig_sf':  ('mu_trig_sf', 'mu_trig_sfup', 'mu_trig_sfdown'),
    'btag_bc_corr':      ('btag_sf', 'btag_sf_bc_correlated_up', 'btag_sf_bc_correlated_down'),
    'btag_bc_uncorr':    ('btag_sf', 'btag_sf_bc_uncorrelated_up', 'btag_sf_bc_uncorrelated_down'),
    'btag_light_corr':   ('btag_sf', 'btag_sf_light_correlated_up', 'btag_sf_light_correlated_down'),
    'btag_light_uncorr': ('btag_sf', 'btag_sf_light_uncorrelated_up', 'btag_sf_light_uncorrelated_down'),
    'puWeight':    ('puWeight', 'puWeight_up', 'puWeight_down'),
    'isr':         (None, 'isr_up', 'isr_down'),
    'fsr':         (None, 'fsr_up', 'fsr_down'),
    'mu_r':        (None, 'mu_r_up', 'mu_r_down'),
    'mu_f':        (None, 'mu_f_up', 'mu_f_down'),
    'mu_rf':       (None, 'mu_rf_up', 'mu_rf_down'),
}
weight_vars = ['genWeight', 'norm_weight'] + sorted({c for t in weight_syst_mapping.values() for c in t if c}
                                                    | set(NOMINAL_FACTORS))

validation_vars = [
    'nPV', 'nPVGood',
    'MET_pt', 'MET_phi', 'lep_pt', 'ele_pt', 'muon_pt',
    'lep_eta', 'ele_eta', 'muon_eta', 'n_ak4', 'n_bjet', 'n_ak8',
    'jet1_pt', 'jet2_pt', 'bjet1_pt',
    'jet1_eta', 'jet2_eta', 'bjet1_eta',
    'jet1_btag', 'jet2_btag', 'bjet1_btag',
    'fatjet1_pt', 'fatjet1_eta', 'fatjet1_mass',
    'ZH_pt', 'ZH_M', 'ZH_bbvLscore',
]
cut_vars = ['ZH_bbvLscore', 'ZH_M', 'ZH_pt', 'n_b_outZH', 'n_ak4jets', 'MET_pt', 'ele_pt', 'muon_pt']
SIGNAL_TRUTH_FLAG = {'ttH': 'Hbb', 'ttZ': 'Zbb'}   # events_Hbb / events_Zbb (genmatcher.match_gen_sig)

binning_dict = {
    'nPV': (40, 0, 80), 'nPVGood': (40, 0, 80),
    'MET_pt': (40, 0, 500), 'MET_phi': (30, -3.14, 3.14),
    'lep_pt': (40, 0, 500), 'ele_pt': (40, 0, 500), 'muon_pt': (40, 0, 500),
    'lep_eta': (30, -2.5, 2.5), 'ele_eta': (30, -2.5, 2.5), 'muon_eta': (30, -2.4, 2.4),
    'n_ak4': (15, 0, 15), 'n_bjet': (10, 0, 10), 'n_ak8': (5, 0, 5),
    'jet1_pt': (40, 0, 500), 'jet2_pt': (40, 0, 500), 'bjet1_pt': (40, 0, 500),
    'jet1_eta': (30, -2.5, 2.5), 'jet2_eta': (30, -2.5, 2.5), 'bjet1_eta': (30, -2.5, 2.5),
    'jet1_btag': (20, 0, 1), 'jet2_btag': (20, 0, 1), 'bjet1_btag': (20, 0, 1),
    'fatjet1_pt': (40, 200, 500), 'fatjet1_eta': (30, -2.5, 2.5), 'fatjet1_mass': (40, 0, 200),
    'ZH_pt': (40, 200, 600), 'ZH_M': (30, 50, 200), 'ZH_bbvLscore': (30, 0, 1),
}
default_binning = (40, 0, 500)


def getZhbbWeight(df_):
    """norm_weight * sign(genWeight) * the nominal SFs (norm_weight already divided by sum sign(genWeight))."""
    weight = df_['norm_weight'] * np.sign(df_['genWeight'])
    for sf in NOMINAL_FACTORS:
        if sf in df_:
            weight = weight * df_[sf].fillna(1.0)
    return weight

In [ ]:
# ==========================================
# 3. DATA EXTRACTION & HISTOGRAMMING
# ==========================================
# The outputs are read one file at a time and histogrammed directly to save RAM.
# hist_dict[var][proc][variation] = {'counts', 'err2'}
hist_dict = {v: {p: {} for p in all_processes} for v in validation_vars}
raw_counts = {p: 0 for p in all_processes}
signal_frames = []          # small per-event tables for the non-bb signal check
missing = set()


def bins_of(var):
    n_bins, x_min, x_max = binning_dict.get(var, default_binning)
    return np.linspace(x_min, x_max, n_bins + 1)


def add_hist(var, proc, syst, vals, weights):
    bins = bins_of(var)
    vals = np.clip(vals, None, bins[-1] - 1e-9)       # overflow into the last bin
    counts, _ = np.histogram(vals, bins=bins, weights=weights)
    err2, _ = np.histogram(vals, bins=bins, weights=weights ** 2)
    h = hist_dict[var][proc].get(syst)
    if h is None:
        hist_dict[var][proc][syst] = {'counts': counts, 'err2': err2}
    else:
        h['counts'] = h['counts'] + counts
        h['err2'] = h['err2'] + err2


def fill_histograms(df, proc):
    weights = df['tot_weight'].to_numpy()
    for var in validation_vars:
        if var not in df:
            missing.add(var)
            continue
        x = df[var].to_numpy()
        mask = x > -998                                 # -999 = not defined (e.g. no electron)
        add_hist(var, proc, 'nominal', x[mask], weights[mask])
        if proc == data_process:
            continue
        # weight systematics (nominal MC only)
        for w_syst, (nom_col, up_col, dn_col) in weight_syst_mapping.items():
            if up_col not in df or dn_col not in df:
                continue
            nom = df[nom_col].to_numpy() if nom_col else np.ones(len(df))
            safe = np.where(nom == 0, 1.0, nom)
            up_w = weights * df[up_col].to_numpy() / safe
            dn_w = weights * df[dn_col].to_numpy() / safe
            add_hist(var, proc, f'{w_syst}Up', x[mask], up_w[mask])
            add_hist(var, proc, f'{w_syst}Down', x[mask], dn_w[mask])


print("\n--- Starting Data Extraction & Histogramming ---")
for group in GROUPS:
    path = os.path.join(COFFEA_DIR, group, "output_all.coffea")
    if not os.path.exists(path):
        print(f"🚨 WARNING: {path} not found; skipping")
        continue
    print(f"Reading {path}")
    filein = load(path)
    genweights = filein.get('sum_signOf_genweights', {})
    for raw_proc, datasets in filein['columns'].items():
        proc = map_process(raw_proc)
        if proc is None:
            continue
        for dataset, cats in datasets.items():
            if TARGET_ERA not in dataset:
                continue
            cols = cats.get(CATEGORY, {}).get('nominal')
            if not cols:
                continue
            names = set(validation_vars) | set(cut_vars)
            if proc != data_process:
                names |= set(weight_vars)
            if proc in sig_processes:
                names |= set(SIGNAL_TRUTH_FLAG.values())
            df = pd.DataFrame({n: cols[f'events_{n}'].value for n in names if f'events_{n}' in cols})
            if proc == data_process:
                df['tot_weight'] = 1.0
            else:
                if dataset not in genweights:
                    print(f"🚨 FATAL: no sum_signOf_genweights for {dataset}; skipping it")
                    continue
                df['norm_weight'] = df['norm_weight'] / genweights[dataset]
                df['tot_weight'] = getZhbbWeight(df) * (TTB_SCALE if proc == 'tt_B' else 1.0)
            df = df[cuts(df)]
            raw_counts[proc] += len(df)
            fill_histograms(df, proc)
            if proc in sig_processes:
                keep = ['tot_weight', 'n_b_outZH', 'MET_pt', 'ZH_bbvLscore'] + \
                       [f for f in SIGNAL_TRUTH_FLAG.values() if f in df]
                sig = df[keep].copy()
                sig['proc'], sig['dataset'] = proc, dataset
                signal_frames.append(sig)
    del filein

if missing:
    print(f"Variables not in the outputs (skipped): {sorted(missing)}")

# ==================================================
# DIAGNOSTIC: PRINT RAW UNWEIGHTED YIELDS
# ==================================================
print("\n" + "=" * 50)
print(f" RAW UNWEIGHTED EVENT COUNTS (PASSING CUTS, {OUTPUT_TAG})")
print("=" * 50)
for proc in all_processes:
    print(f" {proc:<15} : {raw_counts[proc]:,} events")
print("-" * 50)
print(f" {'Total MC':<15} : {sum(raw_counts[p] for p in all_processes if p != data_process):,} events")
print("=" * 50 + "\n")

In [ ]:
# ==========================================
# NON-BB SIGNAL CHECK
# ==========================================
def _decay_from_dataset(name, proc):
    """1.0 = bb-only sample, 0.0 = non-bb sample, NaN = mixed (needs the event flag)."""
    n = name.lower()
    if proc == 'ttH':
        if 'htonon2b' in n:      # TTH-HtoNon2B
            return 0.0
        if 'hto2b' in n:         # TTH-Hto2B
            return 1.0
    if proc == 'ttZ':
        if 'ttll' in n or 'ttnunu' in n:
            return 0.0
    return np.nan          # e.g. TTZ-ZtoQQ contains Z->bb and Z->qq


def check_signal_decays(frames):
    print("\n" + "=" * 70)
    print(" NON-BB SIGNAL CHECK")
    print("=" * 70)
    if not frames:
        print(" no signal events")
        return
    all_sig = pd.concat(frames, ignore_index=True)
    for proc in sig_processes:
        df = all_sig[all_sig['proc'] == proc]
        if df.empty:
            print(f" {proc}: no events, skipping")
            continue
        flag = SIGNAL_TRUTH_FLAG[proc]
        from_name = df['dataset'].map(lambda d: _decay_from_dataset(d, proc)).astype(float)
        if flag in df:
            from_flag = df[flag].astype(float)
            is_bb = from_flag.where(from_flag.notna(), from_name)
            both = from_name.notna() & from_flag.notna()
            mismatch = both & (from_name != from_flag)
            if mismatch.any():
                print(f" 🚨 {proc}: {mismatch.sum()} events where events_{flag} disagrees with the sample name:")
                print(df.loc[mismatch, 'dataset'].value_counts().to_string())
        else:
            print(f" [WARNING] events_{flag} not found for {proc}; using sample names only")
            is_bb = from_name

        label = f'Analysis selection (Xbb {ANALYSIS_XBB_WP})' if ANALYSIS_XBB_WP else 'Analysis selection (no Xbb cut)'
        for sel_name, sel in [('Validation selection', pd.Series(True, index=df.index)),
                              (label, analysis_cuts(df))]:
            sub, bb = df[sel], is_bb[sel]
            cat = np.select([bb == 1.0, bb == 0.0], ['bb', 'non-bb'], 'unknown')
            table = (pd.DataFrame({'dataset': sub['dataset'].values, 'cat': cat, 'w': sub['tot_weight'].values})
                     .pivot_table(index='dataset', columns='cat', values='w', aggfunc='sum', fill_value=0.0))
            for c in ['bb', 'non-bb', 'unknown']:
                if c not in table.columns:
                    table[c] = 0.0
            table = table[['bb', 'non-bb', 'unknown']]
            table['raw events'] = sub['dataset'].value_counts().reindex(table.index).fillna(0).astype(int)
            tot = table[['bb', 'non-bb', 'unknown']].sum()
            total = tot.sum()
            print(f"\n --- {proc}: {sel_name} (weighted yields) ---")
            print(table.round(3).to_string())
            if total != 0:
                print(f" Total: bb = {tot['bb']:.3f}, non-bb = {tot['non-bb']:.3f} "
                      f"({tot['non-bb'] / total:.2%}), unknown = {tot['unknown']:.3f} ({tot['unknown'] / total:.2%})")
    print("=" * 70 + "\n")


check_signal_decays(signal_frames)

In [ ]:
# ==========================================
# 4. PLOTTING FUNCTIONS
# ==========================================
def plot_variables_to_pdf(var_names, hist_dictionary, syst_base_list, output_filename="Data_MC_Plots.pdf"):
    mc_processes = bkg_processes + sig_processes
    var_names = [v for v in var_names if any(hist_dictionary[v][p] for p in all_processes)]

    with PdfPages(output_filename) as pdf:
        for chunk_start in range(0, len(var_names), 9):
            chunk_vars = var_names[chunk_start: chunk_start + 9]
            fig = plt.figure(figsize=(24, 24))
            outer_grid = fig.add_gridspec(3, 3, wspace=0.3, hspace=0.3)

            for idx, var_name in enumerate(chunk_vars):
                row, col = idx // 3, idx % 3
                inner_grid = outer_grid[row, col].subgridspec(2, 1, height_ratios=[3, 1], hspace=0.00)
                ax = fig.add_subplot(inner_grid[0])
                rax = fig.add_subplot(inner_grid[1], sharex=ax)
                ax.tick_params(labelbottom=False)

                bins = bins_of(var_name)
                n_bins = len(bins) - 1
                bin_centers = 0.5 * (bins[1:] + bins[:-1])

                mc_hists, mc_labels, mc_colors_list = [], [], []
                total_mc_counts = np.zeros(n_bins)
                total_mc_stat_err2 = np.zeros(n_bins)
                total_mc_syst_err2 = np.zeros(n_bins)

                # Stack nominal
                for proc in mc_processes:
                    nom_data = hist_dictionary[var_name][proc].get('nominal')
                    if not nom_data:
                        continue
                    counts, err2 = nom_data['counts'], nom_data['err2']
                    mc_hists.append(counts)
                    mc_labels.append(f"{process_labels.get(proc, proc)} ({np.sum(counts):.1f} ± {np.sqrt(np.sum(err2)):.1f})")
                    mc_colors_list.append(mc_colors[proc])
                    total_mc_counts += counts
                    total_mc_stat_err2 += err2

                # Systematics envelope
                for s_base in syst_base_list:
                    syst_up_diff, syst_dn_diff = np.zeros(n_bins), np.zeros(n_bins)
                    for proc in mc_processes:
                        nom_data = hist_dictionary[var_name][proc].get('nominal')
                        if not nom_data:
                            continue
                        nom = nom_data['counts']
                        up = hist_dictionary[var_name][proc].get(f'{s_base}Up', {'counts': nom})['counts']
                        dn = hist_dictionary[var_name][proc].get(f'{s_base}Down', {'counts': nom})['counts']
                        syst_up_diff += (up - nom)
                        syst_dn_diff += (dn - nom)
                    max_diff = np.max(np.abs(syst_up_diff))
                    nom_yield = np.sum(total_mc_counts)
                    if nom_yield > 0 and max_diff > (nom_yield * 0.1):
                        print(f"🚨 WARNING: '{s_base}' causing massive shift on '{var_name}'! Max bin diff: {max_diff:.1f}")
                    total_mc_syst_err2 += np.maximum(np.abs(syst_up_diff), np.abs(syst_dn_diff)) ** 2

                total_mc_err = np.sqrt(total_mc_stat_err2 + total_mc_syst_err2)

                if mc_hists:
                    hep.histplot(mc_hists, bins=bins, ax=ax, stack=True, histtype='fill',
                                 label=mc_labels, color=mc_colors_list, sort='yield')
                ax.stairs(values=total_mc_counts + total_mc_err,
                          baseline=np.clip(total_mc_counts - total_mc_err, 0, None),
                          edges=bins, fill=True, hatch='////', edgecolor='red', facecolor='none',
                          label=r'Stat $\oplus$ Syst Unc.')

                # Data (with blinding)
                data_hist = hist_dictionary[var_name][data_process].get('nominal')
                data_counts = (data_hist['counts'] if data_hist else np.zeros(n_bins)).astype(float)
                if var_name in BLIND:
                    data_counts[bin_centers > BLIND[var_name]] = np.nan
                data_err = np.sqrt(data_counts)
                data_yield = np.nansum(data_counts)
                if data_yield > 0:
                    hep.histplot(data_counts, bins=bins, ax=ax, stack=False, histtype='errorbar',
                                 color='black', label=f"Data ({data_yield:.0f} ± {np.sqrt(data_yield):.1f})",
                                 yerr=data_err)

                # Ratio panel
                with np.errstate(divide='ignore', invalid='ignore'):
                    ratio = data_counts / total_mc_counts
                    ratio_err = np.abs(data_err / total_mc_counts)   # abs: negative MC sums of weights
                    mc_rel_err = np.abs(total_mc_err / total_mc_counts)
                for arr in [ratio, ratio_err, mc_rel_err]:
                    arr[np.isnan(arr) | np.isinf(arr)] = 0
                yerr_down = np.clip(ratio_err, 0, np.maximum(ratio, 0))

                rax.stairs(values=1 + mc_rel_err, baseline=np.clip(1 - mc_rel_err, 0, None),
                           edges=bins, fill=True, hatch='////', edgecolor='red', facecolor='none')
                shown = ratio > 0
                rax.errorbar(bin_centers[shown], ratio[shown], yerr=[yerr_down[shown], ratio_err[shown]],
                             fmt='ko', markersize=3)
                rax.axhline(1, color='black', linestyle='--')

                # Styling
                ax.set_ylabel("Events")
                ax.legend(loc='upper right', ncol=2, fontsize=10)
                finite = data_counts[np.isfinite(data_counts)]
                max_val = max(np.max(total_mc_counts), np.max(finite) if len(finite) else 0)
                ax.set_ylim(0.1, max_val * 100 if max_val > 0 else 100)
                ax.set_yscale('log')
                rax.set_xlabel(var_name)
                rax.set_ylabel("Data / MC")
                rax.set_ylim(0, 2.5)
                hep.cms.label("Preliminary", data=(data_yield > 0), lumi=LUMI, ax=ax, com=13.6, fontsize=12)

            pdf.savefig(fig, bbox_inches='tight')
            plt.close(fig)
    print(f"\nFinished generating plots. Saved to {output_filename}")


def plot_pileup_variations(hist_dictionary, output_filename="Pileup_Variations.pdf"):
    print("\n--- Generating Pileup Overlay Plots ---")
    mc_processes = bkg_processes + sig_processes
    with PdfPages(output_filename) as pdf:
        for var_name in ['nPV', 'nPVGood']:
            if var_name not in hist_dictionary:
                continue
            bins = bins_of(var_name)
            n_bins = len(bins) - 1
            bin_centers = 0.5 * (bins[1:] + bins[:-1])
            nom_counts, up_counts, dn_counts = np.zeros(n_bins), np.zeros(n_bins), np.zeros(n_bins)
            for proc in mc_processes:
                h = hist_dictionary[var_name][proc]
                if 'nominal' in h:
                    nom_counts += h['nominal']['counts']
                    up_counts += h.get('puWeightUp', h['nominal'])['counts']
                    dn_counts += h.get('puWeightDown', h['nominal'])['counts']
            if not nom_counts.any():
                continue

            fig = plt.figure(figsize=(10, 10))
            gs = fig.add_gridspec(2, 1, height_ratios=[3, 1], hspace=0.05)
            ax = fig.add_subplot(gs[0])
            rax = fig.add_subplot(gs[1], sharex=ax)
            ax.tick_params(labelbottom=False)
            hep.histplot(nom_counts, bins=bins, ax=ax, label=f"Nominal (Yield: {np.sum(nom_counts):.1f})", color='black', histtype='step', linewidth=2)
            hep.histplot(up_counts, bins=bins, ax=ax, label=f"PU Up (Yield: {np.sum(up_counts):.1f})", color='red', histtype='step', linewidth=2)
            hep.histplot(dn_counts, bins=bins, ax=ax, label=f"PU Down (Yield: {np.sum(dn_counts):.1f})", color='blue', histtype='step', linewidth=2)
            ax.set_ylabel("Events", fontsize=16)
            ax.legend(loc='upper right', fontsize=14)
            hep.cms.label("Preliminary", data=False, lumi=LUMI, ax=ax, com=13.6, fontsize=16)
            with np.errstate(divide='ignore', invalid='ignore'):
                ratio_up = np.where(nom_counts > 0, up_counts / nom_counts, 0)
                ratio_dn = np.where(nom_counts > 0, dn_counts / nom_counts, 0)
            rax.plot(bin_centers, ratio_up, 'r.-', label="Up / Nom", markersize=8)
            rax.plot(bin_centers, ratio_dn, 'b.-', label="Down / Nom", markersize=8)
            rax.axhline(1, color='black', linestyle='--')
            rax.set_xlabel(var_name, fontsize=16)
            rax.set_ylabel("Var / Nom", fontsize=14)
            rax.set_ylim(0.5, 1.5)
            rax.legend(loc='upper right', fontsize=10, ncol=2)
            pdf.savefig(fig, bbox_inches='tight')
            plt.close(fig)
    print(f"Saved pileup comparison plots to {output_filename}")


def optimize_tagger_wp(hist_dictionary, tagger_var='ZH_bbvLscore', sig_procs=('ttZ', 'ttH'), bkg_procs=None):
    """S/sqrt(B) scan of a cut on the tagger (from the binned nominal histograms)."""
    bkg_procs = bkg_procs or bkg_processes
    print(f"\n--- Optimizing Working Point for {tagger_var} ---")
    bins = bins_of(tagger_var)
    n_bins = len(bins) - 1
    nom_sig, nom_bkg = np.zeros(n_bins), np.zeros(n_bins)
    for proc in sig_procs:
        if 'nominal' in hist_dictionary[tagger_var][proc]:
            nom_sig += hist_dictionary[tagger_var][proc]['nominal']['counts']
    for proc in bkg_procs:
        if 'nominal' in hist_dictionary[tagger_var][proc]:
            nom_bkg += hist_dictionary[tagger_var][proc]['nominal']['counts']
    sig_passing = np.cumsum(nom_sig[::-1])[::-1]
    bkg_passing = np.cumsum(nom_bkg[::-1])[::-1]
    with np.errstate(divide='ignore', invalid='ignore'):
        s_sqrt_b = np.where(bkg_passing > 0, sig_passing / np.sqrt(bkg_passing), 0)
    best_idx = int(np.nanargmax(s_sqrt_b))
    best_threshold = bins[best_idx]
    print(f"Optimal Threshold: > {best_threshold:.3f}")
    print(f"Max S/sqrt(B):     {s_sqrt_b[best_idx]:.3f}")
    print(f"Signal Efficiency: {sig_passing[best_idx] / sig_passing[0] if sig_passing[0] > 0 else 0:.2%}")
    print(f"Overall Bkg Mistag: {bkg_passing[best_idx] / bkg_passing[0] if bkg_passing[0] > 0 else 0:.2%}")
    print(f"Background Yield:  {bkg_passing[best_idx]:.1f} events remaining")

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot(bins[:-1], s_sqrt_b, 'k-', lw=2, label=r'$S/\sqrt{B}$')
    ax.axvline(best_threshold, color='r', linestyle='--', label=f'Optimal WP: > {best_threshold:.2f}')
    ax.set_xlabel(f"{tagger_var} Cut Threshold")
    ax.set_ylabel(r"$S/\sqrt{B}$")
    ax.legend(loc='upper left')
    hep.cms.label("Preliminary", data=False, lumi=LUMI, ax=ax, com=13.6)
    plt.tight_layout()
    plt.savefig(f"WP_Optimization_{tagger_var}_{OUTPUT_TAG}.pdf")
    plt.close()
    return best_threshold

In [ ]:
# ==========================================
# 5. EXECUTE
# ==========================================
all_syst_bases = list(weight_syst_mapping.keys())

plot_pileup_variations(hist_dict, f"Pileup_Variations_{OUTPUT_TAG}.pdf")
#optimal_cut = optimize_tagger_wp(hist_dict, tagger_var='ZH_bbvLscore')

plot_variables_to_pdf(validation_vars, hist_dict, all_syst_bases, f"Data_MC_WeightSys_{OUTPUT_TAG}.pdf")